In [77]:
from pathlib import Path

import io
import onnx
import torch
from onnxsim import simplify

from hailo_sdk_client import ClientRunner

import sys
sys.path.append('..')
from model import Model

In [78]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Generating forward-only graph

In [79]:
# Create a classifier instance
device = "cpu"
batch_size = 8
num_classes = 100
channels = 3
img_h, img_w = 128, 128

optimization_level = -100  # -100, 0-4
compression_level = 0   # 0-5

model_name = 'mobilenetv3_large_100' # resnet18, resnet34, mobilenetv2_100, efficientnet_b2, visformer_tiny
model_frozen_layer_num = {
    'resnet18': 12,
    'resnet34': 20,
    'mobilenetv2_100': 21,
    'efficientnet_b2': 27,
    'visformer_tiny': 24,
    'lcnet_100': 19,
    'semnasnet_100': 20,
    'mobilenetv3_large_100': 22
}
frozen_layer_num = model_frozen_layer_num[model_name]

# artifacts path
artifacts_path = Path(f'../artifacts/{model_name}/artifacts_optim_{optimization_level}_comp_{compression_level}/frozen_layer_{frozen_layer_num}_classes_{num_classes}')
artifacts_path.mkdir(parents=True, exist_ok=True)

pt_model = Model(model_name=model_name, num_classes=num_classes, channels=channels, frozen_layer_num=frozen_layer_num, img_size=img_h)
print(f'Model trainable parameters: {count_parameters(pt_model)}')

for param in pt_model.named_parameters():
    if 'frozen_features' in param[0] or 'bn' in param[0] or 'downsample.1' in param[0]:
        param[1].requires_grad = False
    else:
        param[1].requires_grad = True

torch.save(pt_model, artifacts_path / f'{model_name}.pth')
pt_model = torch.load(artifacts_path / f'{model_name}.pth')
for param in pt_model.named_parameters():
    if param[1].requires_grad:
        print(param[0].ljust(40), '->\t', param[1].requires_grad)

print(f'Model trainable parameters: {count_parameters(pt_model)}')

2024-09-20 12:08:28,232 timm.models._builder [INFO] - Loading pretrained weights from Hugging Face hub (timm/mobilenetv3_large_100.ra_in1k)
2024-09-20 12:08:28,240 urllib3.connectionpool [DEBUG] - Resetting dropped connection: huggingface.co
2024-09-20 12:08:28,500 urllib3.connectionpool [DEBUG] - https://huggingface.co:443 "HEAD /timm/mobilenetv3_large_100.ra_in1k/resolve/main/model.safetensors HTTP/11" 302 0
2024-09-20 12:08:28,509 timm.models._hub [INFO] - [timm/mobilenetv3_large_100.ra_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


Number of layers: 24
Requires grad:
	1.   Conv2d:                   True	->	frozen
	2.   BatchNormAct2d:           True	->	frozen
	3.   DepthwiseSeparableConv:   True	->	frozen
	4.   InvertedResidual:         True	->	frozen
	5.   InvertedResidual:         True	->	frozen
	6.   InvertedResidual:         True	->	frozen
	7.   InvertedResidual:         True	->	frozen
	8.   InvertedResidual:         True	->	frozen
	9.   InvertedResidual:         True	->	frozen
	10.  InvertedResidual:         True	->	frozen
	11.  InvertedResidual:         True	->	frozen
	12.  InvertedResidual:         True	->	frozen
	13.  InvertedResidual:         True	->	frozen
	14.  InvertedResidual:         True	->	frozen
	15.  InvertedResidual:         True	->	frozen
	16.  InvertedResidual:         True	->	frozen
	17.  InvertedResidual:         True	->	frozen
	18.  ConvBnAct:                True	->	frozen
	19.  SelectAdaptivePool2d:     False	->	frozen
	20.  Conv2d:                   True	->	frozen
	21.  Identity:        

/tmp/ipykernel_116998/1293704406.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pt_model = torch.load(artifacts_path / f'{model_name}.pth')


In [80]:
# Generate a random input.
model_inputs = (torch.randn(batch_size, channels, img_h, img_w, device='cpu'),)

model_outputs = pt_model(*model_inputs)
if isinstance(model_outputs, torch.Tensor):
    model_outputs = [model_outputs]
    
input_names = ["input"]
output_names = ["output"]
dynamic_axes = {"input": {0: "batch_size"}, "output": {0: "batch_size"}}

f = io.BytesIO()
torch.onnx.export(
    pt_model,
    model_inputs,
    f,
    input_names=input_names,
    output_names=output_names,
    opset_version=17,
    do_constant_folding=False,
    # training=torch.onnx.TrainingMode.PRESERVE,
    dynamic_axes=dynamic_axes,
    export_params=True,
    # keep_initializers_as_inputs=False,
)

onnx_model = onnx.load_model_from_string(f.getvalue())
print(f'Simplifying ONNX {model_name} model...')
eval_model, check = simplify(onnx_model)
onnx.save(onnx_model, artifacts_path / f'{model_name}.onnx')

Simplifying ONNX mobilenetv3_large_100 model...


#### Parsing ONNX to Hailo

In [81]:
chosen_hw_arch = 'hailo8'
runner = ClientRunner(hw_arch=chosen_hw_arch)

model_path = artifacts_path / f'{model_name}.onnx'
start_node_names = ['input']
# if frozen_layer_num == 12:
#     end_node_names = ['/frozen_features/frozen_features.11/Add']
# elif frozen_layer_num == 3:
#     end_node_names = ['/frozen_features/frozen_features.3/MaxPool']
# else:
#     end_node_names = [f'/frozen_features/frozen_features.{frozen_layer_num}/Add']
model_end_node_names = {
    'resnet18': ['/frozen_features/frozen_features.12/flatten/Flatten'],
    'resnet34': ['/frozen_features/frozen_features.20/flatten/Flatten'],
    'mobilenetv2_100': ['/frozen_features/frozen_features.21/flatten/Flatten'],
    'efficientnet_b2': ['/frozen_features/frozen_features.27/flatten/Flatten'],
    'visformer_tiny': [''],
    'lcnet_100': ['/frozen_features/frozen_features.19/Flatten'],
    'semnasnet_100': ['/frozen_features/frozen_features.20/flatten/Flatten'],
    'mobilenetv3_large_100': ['/frozen_features/frozen_features.22/Flatten'],
}
end_node_names = model_end_node_names[model_name]
net_input_shapes = {'input': [1, channels, img_h, img_w],}

In [82]:
hn, npz = runner.translate_onnx_model(
    str(model_path),
    model_name,
    start_node_names=start_node_names,
    end_node_names=end_node_names,
    net_input_shapes=net_input_shapes,
)

[info] Translation started on ONNX model mobilenetv3_large_100
[info] Restored ONNX model mobilenetv3_large_100 (completion time: 00:00:00.25)
[info] Extracted ONNXRuntime meta-data for Hailo model (completion time: 00:00:01.59)
[info] Translation completed on ONNX model mobilenetv3_large_100 (completion time: 00:00:06.40)
[info] Initialized runner for mobilenetv3_large_100


In [83]:
hailo_har_model_path = artifacts_path / f'{model_name}.har'
runner.save_har(hailo_har_model_path)

[info] Saved HAR to: /home/mateusz/Research/accelerated-cl/artifacts/mobilenetv3_large_100/artifacts_optim_-100_comp_0/frozen_layer_22_classes_100/mobilenetv3_large_100.har


In [84]:
# !hailo profiler {hailo_har_model_path}

#### Model optimization

In [85]:
import numpy as np
from tqdm import tqdm
from PIL import Image


images_list = list(Path('../data/ILSVRC2012_img_val/').rglob('*.JPEG'))

img_mean = [123.675, 116.28, 103.53]
img_std = [58.395, 57.12, 57.375]

# calib_dataset = np.random.randint(0, 255, (1024, img_h, img_w, channels))
# calib_dataset = np.random.random((1024, img_h, img_w, channels))

## calib_dataset = np.zeros((len(images_list), img_h, img_w, channels))
# calib_set_size = 1024
# calib_dataset = np.zeros((calib_set_size, img_h, img_w, channels))
# for idx, img_path in enumerate(tqdm(sorted(images_list[:calib_set_size]))):
#     img = np.array(Image.open(img_path).convert('RGB').resize((img_h, img_w)))
#     img = (img.astype(np.float32) - img_mean) / img_std
#     # img = np.transpose(img, (2, 0, 1))
#     img = np.expand_dims(img, axis=0).astype(np.float32)
#     calib_dataset[idx, :, :, :] = img

calib_set_path = Path('./calib_set.npy')
# np.save(calib_set_path, calib_dataset)

calib_dataset = np.load(calib_set_path)
print(f'Calibration dataset shape: {calib_dataset.shape}')
print(f'Calibration dataset size: {calib_dataset.nbytes / 1024**2:.2f} MB')

Calibration dataset shape: (1024, 128, 128, 3)
Calibration dataset size: 384.00 MB


In [86]:
# Call Optimize to perform the optimization process
runner = ClientRunner(hw_arch=chosen_hw_arch, har=str(hailo_har_model_path))

# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)

# Batch size is 8 by default
# alls = 'normalization1 = normalization([123.675, 116.28, 103.53], [58.395, 57.12, 57.375])\n'

model_script_commands = [
    f'model_optimization_flavor(optimization_level={optimization_level}, compression_level={compression_level}, batch_size=8)\n'
]

# Load the model script to ClientRunner so it will be considered on optimization
runner.load_model_script(''.join(model_script_commands))

runner.optimize(calib_dataset)

# Save the result state to a Quantized HAR file
quantized_har_model_path = str(hailo_har_model_path).replace('.har', '_quantized_model.har')
runner.save_har(quantized_har_model_path)

[info] Loading model script commands to mobilenetv3_large_100 from string
[info] Starting Model Optimization
[info] Model received quantization params from the hn
[info] Starting Mixed Precision
[info] Mixed Precision is done (completion time is 00:00:00.25)
[info] Starting Stats Collector
[info] Using dataset with 64 entries for calibration


Calibration: 100%|██████████| 64/64 [01:39<00:00,  1.55s/entries]


[info] Stats Collector is done (completion time is 00:01:42.71)


In [68]:
# runner.analyze_noise(calib_dataset, data_count=16, batch_size=2)

In [69]:
# !hailo profiler {quantized_har_model_path}

#### Quantized model compilation to HEF format

In [70]:
runner = ClientRunner(har=quantized_har_model_path)
hef = runner.compile()

hef_model_path = quantized_har_model_path.replace('har', 'hef')
with open(hef_model_path, 'wb') as f:
    f.write(hef)

[info] Loading network parameters
[info] Starting Hailo allocation and compilation flow
[info] Using Single-context flow
[info] Resources optimization guidelines: Strategy -> GREEDY Objective -> MAX_FPS
[info] Resources optimization params: max_control_utilization=75%, max_compute_utilization=75%, max_compute_16bit_utilization=75%, max_memory_utilization (weights)=75%, max_input_aligner_utilization=75%, max_apu_utilization=75%
[info] Using Single-context flow
[info] Resources optimization guidelines: Strategy -> GREEDY Objective -> MAX_FPS
[info] Resources optimization params: max_control_utilization=75%, max_compute_utilization=75%, max_compute_16bit_utilization=75%, max_memory_utilization (weights)=75%, max_input_aligner_utilization=75%, max_apu_utilization=75%
[info] input_layer1: Pass
[info] output_layer1: Pass
[info] conv2: Pass
[info] dw2_d0: Pass
[info] dw1: Pass
[info] conv3: Pass
[info] conv1: Pass
[info] dw2_d2: Pass
[info] dw2_d1: Pass
[info] dw2_fs: Pass
[info] dw3_sdc: Pas

#### Export to ONNX with HailoOp operator

In [71]:
onnx_model = runner.get_hailo_runtime_model()
onnx_artifacts_path = hef_model_path.replace('.hef', '_hailo.onnx')
onnx_file = onnx.save(onnx_model, onnx_artifacts_path)
print(f'ONNX with HailoOp saved as: {onnx_artifacts_path}')

[warning] GlobalAvgPool output shape might be different from ONNX shape
ONNX with HailoOp saved as: ../artifacts/semnasnet_100/artifacts_optim_2_comp_0/frozen_layer_20_classes_10/semnasnet_100_quantized_model_hailo.onnx


## Generating training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [72]:
import onnx
import onnxruntime.training.onnxblock as onnxblock


onnx_model = onnx.load(onnx_artifacts_path)

In [73]:
# Creating a class with a Loss function
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [74]:
# from pathlib import Path

# artifacts_path = Path('./artifacts/resnet18_frozen_layer_3/hailo/')
# onnx_artifacts_path = artifacts_path / 'resnet18_quantized_model_hailo.onnx'
# onnx_model = onnx.load_model(onnx_artifacts_path)

# for param in onnx_model.graph.initializer:
#     print(param.name)

In [75]:
training_block = CustomTrainingBlock()
for param in onnx_model.graph.initializer:
    if 'frozen_features' in param.name or 'bn' in param.name or 'downsample.1' in param.name:
        print(param.name.ljust(40), '\t--->\tfrozen')
        training_block.requires_grad(param.name, False)
    else:
        print(param.name.ljust(40), '\t--->\tlearnable')
        training_block.requires_grad(param.name, True)

# Building training graph and eval graph
model_params = None
with onnxblock.base(onnx_model):
    build_output = training_block(*[output.name for output in onnx_model.graph.output])
    print('Graph output nodes:', build_output)
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

2024-09-20 12:07:10,686 root [DEBUG] - Building training block CustomTrainingBlock
2024-09-20 12:07:10,689 root [DEBUG] - Building block: CrossEntropyLoss


2024-09-20 12:07:10,708 root [DEBUG] - Building gradient graph for training block CustomTrainingBlock
2024-09-20 12:07:10,739 root [DEBUG] - The loss output is onnx::loss::20. The gradient graph will be built starting from onnx::loss::20_grad.


post_output.weight                       	--->	learnable
post_output.bias                         	--->	learnable


2024-09-20 12:07:10,762 root [DEBUG] - Adding gradient accumulation nodes for training block CustomTrainingBlock


Graph output nodes: ('onnx::loss::20', 'post_output')


2024-09-20 12:07:10,777 root [DEBUG] - Building forward block AdamW
2024-09-20 12:07:10,779 root [DEBUG] - Building block: AdamWOptimizer


In [76]:
# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / 'checkpoint')

ir_version = 9
opset_import_version = 17

training_model.ir_version = ir_version
# print('Simplifying ONNX training model...')
# training_model, check = simplify(training_model)
# assert check, "Simplified ONNX training model could not be validated"artifacts_optim_2_comp_1
onnx.save(training_model, artifacts_path / 'training_model.onnx')

optimizer_model.ir_version = ir_version
optimizer_model.opset_import[1].version = opset_import_version
onnx.save(optimizer_model, artifacts_path / 'optimizer_model.onnx')

eval_model.ir_version = ir_version
# print('Simplifying ONNX eval model...')
# eval_model, check = simplify(eval_model)
# assert check, "Simplified ONNX eval model could not be validated"
onnx.save(eval_model, artifacts_path / 'eval_model.onnx')

print(f'Artifacts saved in {artifacts_path} directory')

Artifacts saved in ../artifacts/semnasnet_100/artifacts_optim_2_comp_0/frozen_layer_20_classes_10 directory
